# Notebook 2: Query and Visualize Data Spatially and Temporally (Folium)

This notebook is the second half of the flow.

You will:

1. Connect to Upstream
2. Select campaign/station/sensor
3. Query measurements
4. Build interactive spatial and spatiotemporal maps with Folium

## Imports

In [ ]:
import json
from datetime import datetime
from getpass import getpass
from pathlib import Path

import folium
import pandas as pd
from branca.colormap import linear
from folium.plugins import TimestampedGeoJson

from upstream.client import UpstreamClient

from utils import safe_items, safe_total

## Load Saved IDs From Notebook 1 (Optional)

If `outputs/flow_context.json` exists, this cell pulls default IDs from it.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
FLOW_CONTEXT_PATH = NOTEBOOK_DIR / "outputs" / "flow_context.json"

saved_context = {}
if FLOW_CONTEXT_PATH.exists():
    saved_context = json.loads(FLOW_CONTEXT_PATH.read_text(encoding="utf-8"))

saved_context

## User Inputs

Edit these values before running query cells.

In [ ]:
UPSTREAM_USERNAME = input("UPSTREAM_USERNAME: " ).strip()
UPSTREAM_PASSWORD = getpass("UPSTREAM_PASSWORD: ")
UPSTREAM_BASE_URL =  "https://dsoinstitute2026.pods.portals.tapis.io"

CAMPAIGN_ID = saved_context.get("campaign_id")
STATION_ID = saved_context.get("station_id")

# Override manually here if needed
# CAMPAIGN_ID = 123
# STATION_ID = 456

SENSOR_ID = None  # set to an int to force a specific sensor
SENSOR_ALIAS_FILTER = None  # e.g. "River Stage"

PAGE_SIZE = 500
MAX_PAGES = 10
GEOJSON_LIMIT = 5000

## Connect

In [ ]:
client = UpstreamClient(
    username=UPSTREAM_USERNAME,
    password=UPSTREAM_PASSWORD,
    base_url=UPSTREAM_BASE_URL,
)

print("Authenticated:", client.authenticate())
print("CAMPAIGN_ID:", CAMPAIGN_ID)
print("STATION_ID:", STATION_ID)

## List Sensors and Choose One

In [ ]:
sensors_page = client.sensors.list(campaign_id=CAMPAIGN_ID, station_id=STATION_ID, limit=100, page=1)
sensors = safe_items(sensors_page)

print("Sensor total:", safe_total(sensors_page))
for sensor in sensors:
    print(sensor.id, getattr(sensor, "alias", None), getattr(sensor, "units", None))

if SENSOR_ID is None:
    if SENSOR_ALIAS_FILTER:
        for sensor in sensors:
            if getattr(sensor, "alias", None) == SENSOR_ALIAS_FILTER:
                SENSOR_ID = sensor.id
                break
    if SENSOR_ID is None and sensors:
        SENSOR_ID = sensors[0].id

print("Selected SENSOR_ID:", SENSOR_ID)

## Query Measurements (Paginated)

In [ ]:
rows = []

for page in range(1, MAX_PAGES + 1):
    page_data = client.list_measurements(
        campaign_id=CAMPAIGN_ID,
        station_id=STATION_ID,
        sensor_id=SENSOR_ID,
        limit=PAGE_SIZE,
        page=page,
    )

    items = safe_items(page_data)
    if not items:
        break

    for item in items:
        rows.append({
            "measurement_id": getattr(item, "id", None),
            "collectiontime": getattr(item, "collectiontime", None),
            "measurementvalue": getattr(item, "measurementvalue", None),
            "geometry": getattr(item, "geometry", None),
        })

    if len(items) < PAGE_SIZE:
        break

measurements_df = pd.DataFrame(rows)
measurements_df.head()

## Query GeoJSON for Mapping

This fetches point geometry and measurement values used by Folium maps.

In [ ]:
geojson = client.get_measurements_geojson(
    campaign_id=CAMPAIGN_ID,
    station_id=STATION_ID,
    sensor_id=SENSOR_ID,
    limit=GEOJSON_LIMIT,
)

features = geojson.get("features", [])
spatial_rows = []

for feature in features:
    geometry = feature.get("geometry", {})
    properties = feature.get("properties", {})
    coords = geometry.get("coordinates", [None, None])

    lon = coords[0] if len(coords) > 0 else None
    lat = coords[1] if len(coords) > 1 else None
    value = properties.get("measurementvalue", properties.get("value"))
    collectiontime = properties.get("collectiontime")

    spatial_rows.append({
        "lon": lon,
        "lat": lat,
        "measurementvalue": value,
        "collectiontime": collectiontime,
    })

spatial_df = pd.DataFrame(spatial_rows)
spatial_df["lon"] = pd.to_numeric(spatial_df["lon"], errors="coerce")
spatial_df["lat"] = pd.to_numeric(spatial_df["lat"], errors="coerce")
spatial_df["measurementvalue"] = pd.to_numeric(spatial_df["measurementvalue"], errors="coerce")
spatial_df["collectiontime"] = pd.to_datetime(spatial_df["collectiontime"], errors="coerce", utc=True)
spatial_df = spatial_df.dropna(subset=["lon", "lat", "measurementvalue", "collectiontime"])

spatial_df.head()

## Interactive Spatial Map (Folium)

In [ ]:
center_lat = spatial_df["lat"].mean()
center_lon = spatial_df["lon"].mean()

m_spatial = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB positron")

vmin = spatial_df["measurementvalue"].min()
vmax = spatial_df["measurementvalue"].max()
colormap = linear.Viridis_09.scale(vmin, vmax)
colormap.caption = "Measurement Value"

for _, row in spatial_df.iterrows():
    color = colormap(row["measurementvalue"])
    popup_text = (
        f"time: {row['collectiontime']}<br>"
        f"value: {row['measurementvalue']}<br>"
        f"lat: {row['lat']:.6f}, lon: {row['lon']:.6f}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=1,
        popup=folium.Popup(popup_text, max_width=320),
    ).add_to(m_spatial)

colormap.add_to(m_spatial)
m_spatial

## Interactive Spatiotemporal Map (Folium Time Slider)

In [ ]:
time_features = []

for _, row in spatial_df.iterrows():
    color = colormap(row["measurementvalue"])
    time_features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row["lon"], row["lat"]],
        },
        "properties": {
            "time": row["collectiontime"].isoformat(),
            "popup": f"value={row['measurementvalue']}<br>{row['collectiontime']}",
            "icon": "circle",
            "iconstyle": {
                "fillColor": color,
                "fillOpacity": 0.85,
                "stroke": "true",
                "radius": 5
            }
        },
    })

time_geojson = {
    "type": "FeatureCollection",
    "features": time_features
}

m_time = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB positron")
TimestampedGeoJson(
    time_geojson,
    period="PT15M",
    duration="PT15M",
    auto_play=False,
    loop=False,
    max_speed=8,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm:ss",
    time_slider_drag_update=True,
).add_to(m_time)

colormap.add_to(m_time)
m_time